# 👗 AI Fashion Styling Recommender System
**Pipeline:** Wardrobe Upload → Image Embeddings → Clothing Categorization → Outfit Generation → Scoring (Color + Similarity + Rules) → RAG → Final Recommendations

---
**Models Used:**
- `ResNet18` – Visual embeddings
- `all-MiniLM-L6-v2` – Fashion rule retrieval (FAISS)
- `google/flan-t5-base` – Explanation generation (RAG)

## 📦 Cell 1 — Install Dependencies

In [1]:
!pip install -q torch torchvision faiss-cpu sentence-transformers transformers \
               opencv-python-headless scikit-learn Pillow numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.6 MB/s eta 0:00:00


## 🔧 Cell 2 — Imports & Global Config

In [2]:
import os, itertools, warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import cv2
from PIL import Image
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
from torchvision import models, transforms

import faiss
from sentence_transformers import SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer

warnings.filterwarnings('ignore')


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[✓] Using device: {DEVICE}')


CATEGORIES = ['tops', 'bottoms', 'shoes', 'accessories']


CATEGORY_KEYWORDS: Dict[str, List[str]] = {
    'tops':        ['top', 'shirt', 'tshirt', 't-shirt', 'blouse', 'jacket',
                    'coat', 'hoodie', 'sweater', 'kurta', 'sweatshirt', 'tank'],
    'bottoms':     ['bottom', 'pant', 'trouser', 'jeans', 'skirt', 'shorts',
                    'legging', 'chinos', 'palazzo'],
    'shoes':       ['shoe', 'boot', 'sneaker', 'sandal', 'heel', 'loafer',
                    'slipper', 'footwear', 'oxford'],
    'accessories': ['bag', 'belt', 'hat', 'cap', 'scarf', 'watch', 'purse',
                    'jewel', 'necklace', 'earring', 'sunglasses', 'accessory'],
}


SCORE_WEIGHTS = {
    'color':      0.35,
    'similarity': 0.30,
    'category':   0.20,
    'occasion':   0.15,
}

print('[✓] Config loaded.')

[✓] Using device: cuda
[✓] Config loaded.


## 🤖 Cell 3 — Model Loaders (Lazy Singletons)

In [3]:
class Models:
    """Lazy-loaded singleton container — models are loaded only once."""
    _resnet          = None
    _transform       = None
    _sentence_model  = None
    _t5_model        = None
    _t5_tokenizer    = None


def load_resnet() -> Tuple[nn.Module, transforms.Compose]:
    """ResNet18 with classification head removed → 512-d feature extractor."""
    if Models._resnet is None:
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        Models._resnet = nn.Sequential(*list(base.children())[:-1])
        Models._resnet.eval().to(DEVICE)
        Models._transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225]),
        ])
        print('[✓] ResNet18 loaded.')
    return Models._resnet, Models._transform


def load_sentence_transformer() -> SentenceTransformer:
    """MiniLM for fashion-rule semantic search."""
    if Models._sentence_model is None:
        Models._sentence_model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
        print('[✓] SentenceTransformer loaded.')
    return Models._sentence_model


def load_t5() -> Tuple[T5ForConditionalGeneration, T5Tokenizer]:
    """FLAN-T5-base for outfit explanation generation."""
    if Models._t5_model is None:
        name = 'google/flan-t5-base'
        Models._t5_tokenizer = T5Tokenizer.from_pretrained(name)
        Models._t5_model = T5ForConditionalGeneration.from_pretrained(name).to(DEVICE)
        Models._t5_model.eval()
        print('[✓] FLAN-T5-base loaded.')
    return Models._t5_model, Models._t5_tokenizer


load_resnet()
load_sentence_transformer()
load_t5()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 170MB/s]


[✓] ResNet18 loaded.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[✓] FLAN-T5-base loaded.


(T5ForConditionalGeneration(
   (shared): Embedding(32128, 768)
   (encoder): T5Stack(
     (embed_tokens): Embedding(32128, 768)
     (block): ModuleList(
       (0): T5Block(
         (layer): ModuleList(
           (0): T5LayerSelfAttention(
             (SelfAttention): T5Attention(
               (q): Linear(in_features=768, out_features=768, bias=False)
               (k): Linear(in_features=768, out_features=768, bias=False)
               (v): Linear(in_features=768, out_features=768, bias=False)
               (o): Linear(in_features=768, out_features=768, bias=False)
               (relative_attention_bias): Embedding(32, 12)
             )
             (layer_norm): T5LayerNorm()
             (dropout): Dropout(p=0.1, inplace=False)
           )
           (1): T5LayerFF(
             (DenseReluDense): T5DenseGatedActDense(
               (wi_0): Linear(in_features=768, out_features=2048, bias=False)
               (wi_1): Linear(in_features=768, out_features=2048, bias=Fals

## 🖼️ Cell 4 — Image Embeddings

In [4]:
def get_embedding(image_path: str) -> np.ndarray:
    """
    Extract a L2-normalised 512-d embedding from a clothing image.

    Args:
        image_path: Path to image file.
    Returns:
        float32 numpy array of shape (512,).
    """
    resnet, transform = load_resnet()
    img    = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        feat = resnet(tensor)

    vec  = feat.squeeze().cpu().numpy().astype(np.float32)
    norm = np.linalg.norm(vec)
    return vec / (norm + 1e-8)


def embedding_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two pre-normalised embeddings (0–1)."""
    return float(np.dot(a, b))


print('[✓] Embedding functions ready.')

[✓] Embedding functions ready.


## 🏷️ Cell 5 — Clothing Categorisation

In [5]:
def categorize_item(image_path: str, manual_label: Optional[str] = None) -> str:
    """
    Assign a category to a clothing item.

    Priority:
      1. manual_label override (if provided)
      2. Filename keyword heuristic
      3. Fallback → 'accessories'

    Args:
        image_path:   Path to the image.
        manual_label: Optional category override string.
    Returns:
        Category string from CATEGORIES.
    """
    if manual_label and manual_label.lower() in CATEGORIES:
        return manual_label.lower()

    stem = Path(image_path).stem.lower().replace('-', '_').replace(' ', '_')

    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in stem for kw in keywords):
            return category

    return 'accessories'


print('[✓] Categoriser ready.')

[✓] Categoriser ready.


## 👔 Cell 6 — Wardrobe System

In [6]:
def build_wardrobe(
    image_paths: List[str],
    manual_labels: Optional[Dict[str, str]] = None,
) -> Dict[str, List[Dict]]:
    """
    Build a structured wardrobe from a list of image paths.

    Each item dict:
        path      – original image path
        category  – inferred category
        embedding – normalised 512-d float32 array

    Args:
        image_paths:   List of clothing image paths.
        manual_labels: {path: category} overrides (optional).
    Returns:
        Wardrobe dict keyed by category.
    """
    wardrobe: Dict[str, List[Dict]] = {cat: [] for cat in CATEGORIES}
    labels = manual_labels or {}

    for path in image_paths:
        if not os.path.exists(path):
            print(f'  [!] Not found, skipping: {path}')
            continue

        category  = categorize_item(path, labels.get(path))
        embedding = get_embedding(path)

        wardrobe[category].append({
            'path':      path,
            'category':  category,
            'embedding': embedding,
        })
        print(f'  [+] {Path(path).name:30s} → {category}')

    total = sum(len(v) for v in wardrobe.values())
    print(f'\n[✓] Wardrobe built: {total} items')
    for cat in CATEGORIES:
        print(f'     {cat:15s}: {len(wardrobe[cat])} items')
    return wardrobe


print('[✓] Wardrobe builder ready.')

[✓] Wardrobe builder ready.


## 🎨 Cell 7 — Color Extraction & Harmony Scoring

In [7]:
def extract_dominant_color(image_path: str, k: int = 3) -> np.ndarray:
    """
    Return the most dominant BGR colour via K-Means clustering.

    Args:
        image_path: Path to image.
        k:          Number of colour clusters.
    Returns:
        BGR array of shape (3,).
    """
    img = cv2.imread(image_path)
    if img is None:
        return np.array([128, 128, 128], dtype=np.float32)

    img    = cv2.resize(img, (100, 100))
    pixels = img.reshape(-1, 3).astype(np.float32)

    kmeans = KMeans(n_clusters=k, n_init=5, random_state=42)
    kmeans.fit(pixels)

    counts   = np.bincount(kmeans.labels_)
    dominant = kmeans.cluster_centers_[np.argmax(counts)]
    return dominant


def _bgr_to_hue(bgr: np.ndarray) -> float:
    """Convert BGR pixel to HSV hue (0–360)."""
    pixel = np.uint8([[bgr]])
    hsv   = cv2.cvtColor(pixel, cv2.COLOR_BGR2HSV)
    return float(hsv[0, 0, 0]) * 2.0


def _bgr_saturation(bgr: np.ndarray) -> float:
    """Return HSV saturation (0–255) for a BGR pixel."""
    pixel = np.uint8([[bgr]])
    return float(cv2.cvtColor(pixel, cv2.COLOR_BGR2HSV)[0, 0, 1])


def _hue_diff(h1: float, h2: float) -> float:
    """Smallest angular difference on a 360° colour wheel."""
    d = abs(h1 - h2) % 360
    return d if d <= 180 else 360 - d


def color_harmony_score(colors: List[np.ndarray]) -> float:
    """
    Score (0–1) based on colour harmony rules.

    Rules (per pair):
        Monochromatic  hue diff < 15°        → 1.00
        Analogous      hue diff < 45°        → 0.80
        Complementary  hue diff 150–210°     → 0.90
        Neutral item   saturation < 30       → 0.85
        Otherwise                            → 0.40

    Returns:
        Mean pairwise harmony score.
    """
    if len(colors) < 2:
        return 1.0

    hues = [_bgr_to_hue(c) for c in colors]
    sats = [_bgr_saturation(c) for c in colors]

    scores = []
    for i, j in itertools.combinations(range(len(colors)), 2):
        diff = _hue_diff(hues[i], hues[j])
        if sats[i] < 30 or sats[j] < 30:
            scores.append(0.85)
        elif diff < 15:
            scores.append(1.0)
        elif diff < 45:
            scores.append(0.80)
        elif 150 <= diff <= 210:
            scores.append(0.90)
        else:
            scores.append(0.40)

    return round(float(np.mean(scores)), 4)


print('[✓] Color functions ready.')

[✓] Color functions ready.


## 👗 Cell 8 — Outfit Combination Generator

In [8]:
def generate_outfits(
    wardrobe: Dict[str, List[Dict]],
    max_outfits: int = 50,
) -> List[Dict]:
    """
    Generate valid (top, bottom, shoe) combos with optional accessory.

    Constraints:
      - Requires ≥1 top, ≥1 bottom, ≥1 shoe.
      - Max 1 accessory per outfit (None = no accessory).
      - Capped at max_outfits to avoid combinatorial explosion.

    Returns:
        List of outfit dicts with keys: top, bottom, shoe, accessory.
    """
    tops        = wardrobe.get('tops', [])
    bottoms     = wardrobe.get('bottoms', [])
    shoes       = wardrobe.get('shoes', [])
    accessories = wardrobe.get('accessories', []) + [None]

    if not (tops and bottoms and shoes):
        print('[!] Need at least 1 top, 1 bottom, 1 shoe to generate outfits.')
        return []

    outfits = []
    for top, bottom, shoe, acc in itertools.product(tops, bottoms, shoes, accessories):
        outfits.append({
            'top':       top,
            'bottom':    bottom,
            'shoe':      shoe,
            'accessory': acc,
        })
        if len(outfits) >= max_outfits:
            break

    print(f'[✓] Generated {len(outfits)} outfit combinations.')
    return outfits


print('[✓] Outfit generator ready.')

[✓] Outfit generator ready.


## ⭐ Cell 9 — Scoring System

In [9]:
def _embedding_cohesion(items: List[Dict]) -> float:
    """Mean pairwise cosine similarity → visual cohesion score (0–1)."""
    embeddings = [item['embedding'] for item in items]
    if len(embeddings) < 2:
        return 1.0
    pairs = list(itertools.combinations(embeddings, 2))
    sims  = [embedding_similarity(a, b) for a, b in pairs]

    return round((float(np.mean(sims)) + 1.0) / 2.0, 4)


def _category_completeness(outfit: Dict) -> float:
    """Reward complete outfits (top + bottom + shoe → 1.0)."""
    filled = sum(1 for k in ['top', 'bottom', 'shoe'] if outfit[k] is not None)
    return filled / 3.0


OCCASION_KEYWORDS: Dict[str, List[str]] = {
    'casual':  ['jeans', 'tshirt', 't-shirt', 'sneaker', 'casual', 'hoodie', 'shorts'],
    'formal':  ['shirt', 'trouser', 'oxford', 'blazer', 'formal', 'suit', 'chinos'],
    'party':   ['dress', 'heel', 'party', 'sequin', 'glitter', 'skirt'],
    'sport':   ['sport', 'gym', 'track', 'athletic', 'shorts', 'legging'],
}


def _occasion_score(outfit: Dict, occasion: str = 'casual') -> float:
    """Keyword-based occasion relevance score (0–1)."""
    keywords = OCCASION_KEYWORDS.get(occasion, [])
    items    = [outfit['top'], outfit['bottom'], outfit['shoe']]
    if outfit['accessory']:
        items.append(outfit['accessory'])

    hits = 0
    for item in items:
        if item:
            stem = Path(item['path']).stem.lower()
            if any(kw in stem for kw in keywords):
                hits += 1

    return round(hits / len(items), 4) if items else 0.5


def score_outfit(
    outfit: Dict,
    occasion: str = 'casual',
    weights: Optional[Dict[str, float]] = None,
) -> Dict[str, float]:
    """
    Compute a weighted composite score (0–1) for an outfit.

    Sub-scores:
        color      – pairwise colour harmony
        similarity – ResNet embedding cohesion
        category   – outfit completeness
        occasion   – keyword-based occasion match

    Returns:
        Dict with individual sub-scores and 'total'.
    """
    w     = weights or SCORE_WEIGHTS
    items = [outfit['top'], outfit['bottom'], outfit['shoe']]
    if outfit['accessory']:
        items.append(outfit['accessory'])

    colors     = [extract_dominant_color(item['path']) for item in items]
    s_color    = color_harmony_score(colors)
    s_sim      = _embedding_cohesion(items)
    s_category = _category_completeness(outfit)
    s_occasion = _occasion_score(outfit, occasion)

    total = (w['color']      * s_color
           + w['similarity'] * s_sim
           + w['category']   * s_category
           + w['occasion']   * s_occasion)

    return {
        'color':      round(s_color,    4),
        'similarity': round(s_sim,      4),
        'category':   round(s_category, 4),
        'occasion':   round(s_occasion, 4),
        'total':      round(float(total), 4),
    }


print('[✓] Scoring system ready.')

[✓] Scoring system ready.


## 📚 Cell 10 — Fashion Knowledge Base (FAISS + RAG)

In [10]:
DEFAULT_FASHION_RULES = [

    'For summer weddings, choose breathable fabrics like cotton, linen, chiffon. Prefer pastel shades.',
    'For formal events, avoid loud prints; choose solid colors, clean silhouettes, minimal accessories.',
    'For a casual brunch, light colors and relaxed fits work well. Sneakers or flats are ideal.',
    'For office wear, stick to neutral colors, modest fits, and structured pieces like blazers or trousers.',
    'If the outfit has a heavy pattern, balance with solid pieces. If solid, add a patterned accessory.',
    'Warm undertones pair well with earthy shades (beige, olive, mustard). Cool undertones suit blues, greys, lavender.',
    'Monochrome outfits look elegant. Use one accent item like a bag or shoes for contrast.',
    'Denim is casual; avoid denim for formal events unless styled with a blazer and dressy footwear.',

    'Pair neutral tops (white, grey, black) with bold-coloured bottoms for a balanced look.',
    'Complementary colours — opposite on the colour wheel — create high-energy, striking outfits.',
    'Analogous colours (adjacent on the wheel) produce harmonious, relaxed combinations.',
    'White and navy is a timeless pairing suitable for both casual and semi-formal occasions.',
    'Slim-fit trousers visually balance an oversized top.',
    'Chunky shoes pair best with wide-leg or relaxed-fit bottoms.',
    'Earth tones (tan, olive, brown, camel) naturally complement each other.',
    'Mixing textures (denim with silk, leather with cotton) adds interest without clashing colours.',
    'For casual looks, white sneakers work with almost any outfit.',
    'Avoid mixing more than three colours in a single outfit.',
    'A belt in the same colour as your shoes creates a polished, put-together look.',
    'Statement accessories work best with simple, solid outfits.',
]


class FashionKnowledgeBase:
    """
    FAISS-backed vector store for fashion rules.
    Supports top-k semantic retrieval.
    """

    def __init__(self, rules: Optional[List[str]] = None):
        self._st    = load_sentence_transformer()
        self._rules = rules if rules is not None else DEFAULT_FASHION_RULES
        self._index = None
        self._build_index()

    def _build_index(self):
        """Encode rules and build a cosine-similarity FAISS index."""
        embs = self._st.encode(
            self._rules,
            normalize_embeddings=True,
            show_progress_bar=False
        ).astype(np.float32)

        dim          = embs.shape[1]
        self._index  = faiss.IndexFlatIP(dim)
        self._index.add(embs)
        print(f'[✓] Knowledge base ready: {len(self._rules)} rules indexed.')

    def retrieve(self, query: str, k: int = 3) -> List[Tuple[str, float]]:
        """
        Retrieve top-k rules most relevant to query.

        Returns:
            List of (rule_text, score) tuples.
        """
        q_emb           = self._st.encode([query], normalize_embeddings=True).astype(np.float32)
        scores, indices = self._index.search(q_emb, k)
        return [(self._rules[i], float(scores[0][rank]))
                for rank, i in enumerate(indices[0])]

    def add_rules(self, new_rules: List[str]):
        """Incrementally add new rules and rebuild the index."""
        self._rules += new_rules
        self._build_index()



KB = FashionKnowledgeBase()

[✓] Knowledge base ready: 20 rules indexed.


## 💬 Cell 11 — RAG Explanation Generator (FLAN-T5)

In [11]:
def retrieve_rules(query: str, k: int = 3) -> List[Tuple[str, float]]:
    """
    Retrieve top-k fashion rules relevant to a query.

    Args:
        query: Natural-language outfit description or question.
        k:     Number of rules to retrieve.
    Returns:
        List of (rule, relevance_score) tuples.
    """
    return KB.retrieve(query, k=k)


def generate_explanation(
    outfit: Dict,
    score_breakdown: Dict[str, float],
    occasion: str = 'casual',
    k_rules: int = 3,
) -> str:
    """
    Generate a natural-language explanation for an outfit using RAG + FLAN-T5.

    Steps:
      1. Build a text description of the outfit from filenames.
      2. Retrieve relevant fashion rules from FAISS.
      3. Build a structured prompt.
      4. Generate explanation via FLAN-T5.

    Args:
        outfit:          Outfit dict from generate_outfits().
        score_breakdown: Output of score_outfit().
        occasion:        Target occasion.
        k_rules:         Number of rules to retrieve.
    Returns:
        AI-generated explanation string.
    """

    def item_name(item):
        return Path(item['path']).stem.replace('_', ' ').replace('-', ' ') if item else 'none'

    desc = (
        f"top: {item_name(outfit['top'])}, "
        f"bottom: {item_name(outfit['bottom'])}, "
        f"shoes: {item_name(outfit['shoe'])}"
    )
    if outfit['accessory']:
        desc += f", accessory: {item_name(outfit['accessory'])}"

    query       = f'{occasion} outfit with {desc}'
    rules       = retrieve_rules(query, k=k_rules)
    rules_text  = '\n'.join([f'Rule {i+1}: {r}' for i, (r, _) in enumerate(rules)])

    prompt = f"""You are a fashion stylist. Use ONLY the rules below to give a short outfit review.

Rules:
{rules_text}

Outfit: {desc}
Occasion: {occasion}
Score: {score_breakdown['total']:.2f}/1.00 \
(color={score_breakdown['color']:.2f}, \
similarity={score_breakdown['similarity']:.2f}, \
category={score_breakdown['category']:.2f}, \
occasion={score_breakdown['occasion']:.2f})

Write 2-3 sentences: why this outfit works, which rules apply, and one tip."""

    t5_model, tokenizer = load_t5()
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.no_grad():
        output = t5_model.generate(
            **inputs,
            max_new_tokens=180,
            min_new_tokens=60,
            num_beams=6,
            repetition_penalty=1.7,
            no_repeat_ngram_size=4,
            length_penalty=1.0,
            early_stopping=True,
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)


print('[✓] RAG explanation generator ready.')

[✓] RAG explanation generator ready.


## 🏆 Cell 12 — Final Recommendation Engine

In [12]:
def recommend_outfits(
    wardrobe: Dict[str, List[Dict]],
    occasion: str = 'casual',
    top_n: int = 3,
    max_outfits: int = 50,
) -> List[Dict]:
    """
    Full pipeline: generate → score → rank → explain → return top N outfits.

    Args:
        wardrobe:    Output of build_wardrobe().
        occasion:    'casual' | 'formal' | 'party' | 'sport'
        top_n:       Number of top outfits to return.
        max_outfits: Max combinations to evaluate.
    Returns:
        List of top_n recommendation dicts, each containing:
            rank, items, score_breakdown, explanation.
    """
    print(f'\n[→] Generating outfits for occasion: "{occasion}"...')
    outfits = generate_outfits(wardrobe, max_outfits=max_outfits)

    if not outfits:
        return []


    print('[→] Scoring outfits...')
    scored = []
    for outfit in outfits:
        breakdown = score_outfit(outfit, occasion=occasion)
        scored.append((outfit, breakdown))


    scored.sort(key=lambda x: x[1]['total'], reverse=True)
    top_outfits = scored[:top_n]

    print(f'[→] Generating AI explanations for top {top_n} outfits...')
    recommendations = []
    for rank, (outfit, breakdown) in enumerate(top_outfits, start=1):
        explanation = generate_explanation(outfit, breakdown, occasion=occasion)
        items = {
            'top':       outfit['top']['path'],
            'bottom':    outfit['bottom']['path'],
            'shoe':      outfit['shoe']['path'],
            'accessory': outfit['accessory']['path'] if outfit['accessory'] else None,
        }
        recommendations.append({
            'rank':            rank,
            'items':           items,
            'score_breakdown': breakdown,
            'explanation':     explanation,
        })
        print(f'  [✓] Outfit #{rank} scored & explained (total={breakdown["total"]:.4f})')

    print('\n[✓] Recommendations ready!')
    return recommendations


print('[✓] Recommendation engine ready.')

[✓] Recommendation engine ready.


## 📊 Cell 13 — Visualisation Helper

In [13]:
def display_recommendations(recommendations: List[Dict], figsize_per_outfit=(16, 4)):
    """
    Display each recommended outfit as a row of images with scores & explanation.

    Args:
        recommendations: Output of recommend_outfits().
    """
    if not recommendations:
        print('[!] No recommendations to display.')
        return

    for rec in recommendations:
        rank  = rec['rank']
        items = rec['items']
        sb    = rec['score_breakdown']
        exp   = rec['explanation']


        slots = [(label, path) for label, path in items.items() if path is not None]
        n     = len(slots)

        fig, axes = plt.subplots(1, n, figsize=figsize_per_outfit)
        if n == 1:
            axes = [axes]

        fig.suptitle(
            f'Outfit #{rank}  |  Score: {sb["total"]:.3f}  '
            f'(color={sb["color"]:.2f}  sim={sb["similarity"]:.2f}  '
            f'cat={sb["category"]:.2f}  occ={sb["occasion"]:.2f})',
            fontsize=13, fontweight='bold', y=1.02
        )

        for ax, (label, path) in zip(axes, slots):
            try:
                img = Image.open(path).convert('RGB')
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5, 'Image\nnot found',
                        ha='center', va='center', transform=ax.transAxes)
            ax.set_title(label.capitalize(), fontsize=11)
            ax.axis('off')

        swatch_ax = fig.add_axes([0.0, -0.08, 1.0, 0.06])
        swatch_ax.axis('off')
        for si, (label, path) in enumerate(slots):
            bgr   = extract_dominant_color(path)
            rgb   = bgr[::-1] / 255.0
            patch = mpatches.FancyBboxPatch(
                (si / n + 0.01, 0.1), 1/n - 0.02, 0.8,
                boxstyle='round,pad=0.02', color=rgb
            )
            swatch_ax.add_patch(patch)
        swatch_ax.set_xlim(0, 1)
        swatch_ax.set_ylim(0, 1)

        plt.tight_layout()
        plt.show()

        print(f'\n💬 AI Explanation:\n{exp}\n')
        print('─' * 80)


print('[✓] Visualiser ready.')

[✓] Visualiser ready.


## 🚀 Cell 14 — Upload Your Wardrobe & Run

> **Naming convention for auto-categorisation:**  
> Include a keyword in the filename:  
> - Tops → `white_shirt.jpg`, `blue_tshirt.png`  
> - Bottoms → `black_jeans.jpg`, `navy_trouser.png`  
> - Shoes → `white_sneaker.jpg`, `brown_boot.png`  
> - Accessories → `leather_belt.jpg`, `red_bag.png`  
>
> Or use the `manual_labels` dict to override.

In [14]:
from google.colab import files

print('Upload your wardrobe images (JPG/PNG):')
uploaded = files.upload()

image_paths = list(uploaded.keys())
print(f'\n[✓] {len(image_paths)} images uploaded: {image_paths}')

Upload your wardrobe images (JPG/PNG):


Saving 12_PANT-FLATLAY_SP1-24_BGOLDSTEIN_2180_91178e67-9314-4a84-a9cd-2bba7177341f.webp to 12_PANT-FLATLAY_SP1-24_BGOLDSTEIN_2180_91178e67-9314-4a84-a9cd-2bba7177341f.webp
Saving 6149mVtxPvL._AC_UY1100_.jpg to 6149mVtxPvL._AC_UY1100_.jpg

[✓] 2 images uploaded: ['12_PANT-FLATLAY_SP1-24_BGOLDSTEIN_2180_91178e67-9314-4a84-a9cd-2bba7177341f.webp', '6149mVtxPvL._AC_UY1100_.jpg']


In [15]:

manual_labels = {}

print('\n=== Building Wardrobe ===')
wardrobe = build_wardrobe(image_paths, manual_labels=manual_labels)


=== Building Wardrobe ===
  [+] 12_PANT-FLATLAY_SP1-24_BGOLDSTEIN_2180_91178e67-9314-4a84-a9cd-2bba7177341f.webp → bottoms
  [+] 6149mVtxPvL._AC_UY1100_.jpg    → accessories

[✓] Wardrobe built: 2 items
     tops           : 0 items
     bottoms        : 1 items
     shoes          : 0 items
     accessories    : 1 items


In [16]:

OCCASION = 'casual'

print(f'\n=== Running Recommendation Engine (occasion="{OCCASION}") ===')
recommendations = recommend_outfits(
    wardrobe,
    occasion=OCCASION,
    top_n=3,
    max_outfits=50,
)


=== Running Recommendation Engine (occasion="casual") ===

[→] Generating outfits for occasion: "casual"...
[!] Need at least 1 top, 1 bottom, 1 shoe to generate outfits.


In [17]:

print('\n=== Top 3 Outfit Recommendations ===')
display_recommendations(recommendations)


=== Top 3 Outfit Recommendations ===
[!] No recommendations to display.


## 🔍 Cell 15 — Bonus: Ask the Fashion RAG Directly

In [18]:
def rag_stylist_answer(user_query: str, k: int = 3) -> str:
    """
    Direct Q&A with the fashion knowledge base.
    Retrieves top-k rules and generates an answer via FLAN-T5.

    Args:
        user_query: Any fashion question.
        k:          Number of rules to retrieve.
    Returns:
        Generated answer string.
    """
    rules      = retrieve_rules(user_query, k=k)
    rules_text = '\n'.join([f'Rule {i+1}: {r}' for i, (r, _) in enumerate(rules)])

    prompt = f"""Use ONLY these rules. Answer clearly in 2-3 sentences.

{rules_text}

Question: {user_query}

Outfit:
Reason: <cite Rule numbers>
Tip: <one actionable tip>"""

    t5_model, tokenizer = load_t5()
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)

    with torch.no_grad():
        output = t5_model.generate(
            **inputs,
            max_new_tokens=220,
            min_new_tokens=90,
            num_beams=6,
            repetition_penalty=1.7,
            no_repeat_ngram_size=4,
            length_penalty=1.0,
            early_stopping=True,
        )

    answer = tokenizer.decode(output[0], skip_special_tokens=True)

    print('Retrieved Rules:')
    for i, (rule, score) in enumerate(rules):
        print(f'  Rule {i+1} (score={score:.3f}): {rule}')
    print(f'\nAI Answer:\n{answer}')
    return answer

_ = rag_stylist_answer('What should I wear to a summer wedding as a college student?')

Retrieved Rules:
  Rule 1 (score=0.624): For summer weddings, choose breathable fabrics like cotton, linen, chiffon. Prefer pastel shades.
  Rule 2 (score=0.479): For a casual brunch, light colors and relaxed fits work well. Sneakers or flats are ideal.
  Rule 3 (score=0.466): For office wear, stick to neutral colors, modest fits, and structured pieces like blazers or trousers.

AI Answer:
Rule 1: For summer weddings, choose breathable fabrics like cotton. Prefer pastel shades and relaxed fits work well; Sneakers or flat shoes are ideal for office wear (rule 3). Answer clearly in 2-3 sentences about what you're going to be wearing on the day of your graduation from high school/collegiate level as an college student who is graduating this fall with no experience at any point during that time period! Tip = "one actionable tip".


## 📈 Cell 16 — Score Breakdown Chart

In [19]:
def plot_score_breakdown(recommendations: List[Dict]):
    """Bar chart comparing sub-scores across top recommendations."""
    if not recommendations:
        print('[!] No recommendations to plot.')
        return

    metrics = ['color', 'similarity', 'category', 'occasion', 'total']
    colors  = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#2C3E50']

    x      = np.arange(len(metrics))
    width  = 0.25
    fig, ax = plt.subplots(figsize=(12, 5))

    for i, rec in enumerate(recommendations):
        sb     = rec['score_breakdown']
        values = [sb[m] for m in metrics]
        offset = (i - len(recommendations)/2 + 0.5) * width
        bars   = ax.bar(x + offset, values, width, label=f"Outfit #{rec['rank']}",
                        alpha=0.85)

        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=8)

    ax.set_xlabel('Score Component', fontsize=12)
    ax.set_ylabel('Score (0–1)', fontsize=12)
    ax.set_title('Score Breakdown — Top Outfit Recommendations', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([m.capitalize() for m in metrics])
    ax.set_ylim(0, 1.15)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_score_breakdown(recommendations)

[!] No recommendations to plot.
